# Cat/dog block geometry with stratified image bootstrap

Four fixed pretrained CNNs, matched 100 cats + 100 dogs. **Primary readout: adjacent linear CKA.** S and held-out probe are secondary; other metrics are descriptive. Each plot uses one model's observed layer index.

Run cells in order in Colab (GPU recommended for extraction). Each feature cache preserves image identities; bootstrap reuses features and performs no network forward passes. The 1,000 class-stratified draws are shared across every layer and all four models. Pointwise p10–p90 intervals cover the central **80%** of bootstrap values; they are not simultaneous curve bands. Positive dip fractions are descriptive, not p-values.

No bootstrap results are present until the execution/export cells finish. Do not mistake old notebook outputs for this run.


Publication: image outputs are deferred; executed code, execution counts and numerical/text outputs are preserved.


In [1]:
import torch, torchvision, numpy as np, pandas as pd, matplotlib.pyplot as plt
from torchvision import models
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from PIL import Image
from io import BytesIO
from pathlib import Path
import zipfile, urllib.request, hashlib, gc

SEED=42; N_PER_CLASS=100; BATCH_SIZE=16; PROBE_SPLITS=5
np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Run in order, releasing each network before loading the next.
MODEL_SPECS=[
 ('ResNet-18',models.resnet18,models.ResNet18_Weights.IMAGENET1K_V1),
 ('ResNet-152',models.resnet152,models.ResNet152_Weights.IMAGENET1K_V2),
 ('ConvNeXt-Tiny',models.convnext_tiny,models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1),
 ('ConvNeXt-Base',models.convnext_base,models.ConvNeXt_Base_Weights.IMAGENET1K_V1),
]
print('torch',torch.__version__,'torchvision',torchvision.__version__,'device',device)

BOOTSTRAP_REPEATS = 1000
BOOTSTRAP_SEED = 20261006
RESULT_DIR = Path("block_geometry_metrics")
RESULT_DIR.mkdir(exist_ok=True)


torch 2.11.0+cu130 torchvision 0.26.0+cu130 device cuda


In [2]:
# 1. Same 100 cats and 100 dogs for every model and every stage.
# Zenodo Cats and Dogs sample: https://zenodo.org/records/5226945
# Fixed ResNet-18 V1 preprocessing for all models; ResNet-152 V2 recommends a different recipe.
transform = MODEL_SPECS[0][2].transforms()
# Use this single transform for every architecture and weight condition.
archive = Path('cats_dogs_light.zip')
if not archive.exists():
    urllib.request.urlretrieve('https://zenodo.org/records/5226945/files/cats_dogs_light.zip?download=1', archive)
expected_md5 = '5e014163374c3bf7069c923de2d619c8'
actual_md5 = hashlib.md5(archive.read_bytes()).hexdigest()
assert actual_md5 == expected_md5, f'Download checksum mismatch: {actual_md5}'

class ZipCatsDogs(Dataset):
    def __init__(self, path, transform):
        self.path, self.transform = path, transform
        with zipfile.ZipFile(path) as zf:
            self.names = [name for name in zf.namelist()
                          if '/test/' in name.lower() and name.lower().endswith(('.jpg', '.jpeg', '.png'))
                          and Path(name).name.lower().startswith(('cat.', 'dog.'))]
        self.targets = [3 if Path(name).name.lower().startswith('cat.') else 5 for name in self.names]
        assert self.names, 'No labeled cat/dog test images found; inspect archive paths.'
    def __len__(self): return len(self.names)
    def __getitem__(self, index):
        with zipfile.ZipFile(self.path) as zf:
            image = Image.open(BytesIO(zf.read(self.names[index]))).convert('RGB')
        return self.transform(image), self.targets[index]

raw = ZipCatsDogs(archive, transform)
y_all = np.asarray(raw.targets)
rng = np.random.default_rng(SEED)
assert all((y_all == cls).sum() >= N_PER_CLASS for cls in (3, 5)), 'Reduce N_PER_CLASS.'
indices = np.concatenate([rng.choice(np.flatnonzero(y_all == cls), N_PER_CLASS, replace=False) for cls in (3, 5)])
indices = rng.permutation(indices).tolist()
selected = Subset(raw, indices)
loader = DataLoader(selected, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(device.type == 'cuda'))
labels = np.asarray([y_all[i] for i in indices])
print(f'{len(selected)} images: {(labels == 3).sum()} cats, {(labels == 5).sum()} dogs')


200 images: 100 cats, 100 dogs


In [3]:
# The hook list follows actual module execution order. Stage boundaries are explicit.
def observed_modules(name,model):
    pairs=[]; boundaries=[]
    if name.startswith('ResNet'):
        pairs.append(('stem',model.maxpool)); boundaries.append('stem')
        for i in range(1,5):
            for j,block in enumerate(getattr(model,f'layer{i}')):
                label=f's{i}.b{j+1}'
                pairs.append((label,block))
                if j==0: boundaries.append(label)
    else:
        for idx,module in enumerate(model.features):
            if idx==0:
                pairs.append(('stem',module)); boundaries.append('stem')
            elif idx%2==0:
                label=f'down{idx//2}'
                pairs.append((label,module)); boundaries.append(label)
            else:
                stage=(idx+1)//2
                for j,block in enumerate(module):
                    label=f's{stage}.b{j+1}'
                    pairs.append((label,block))
                    if j==0: boundaries.append(label)
    return pairs,boundaries

def extract_blocks(model,loader,pairs):
    model=model.to(device).eval()
    record={key:[] for key,_ in pairs}; cache={}; hooks=[]
    for key,module in pairs:
        def grab(m,inp,out,key=key):
            h=out.detach()
            if h.ndim==4: h=h.mean((-2,-1))
            cache[key]=h.flatten(1).cpu()
        hooks.append(module.register_forward_hook(grab))
    try:
        with torch.inference_mode():
            for images,_ in loader:
                _=model(images.to(device))
                for key in record: record[key].append(cache.pop(key))
    finally:
        for h in hooks: h.remove()
        model.cpu()
        if device.type=='cuda': torch.cuda.empty_cache()
    return {key:torch.cat(parts).numpy() for key,parts in record.items()}


In [4]:
# Run after the notebook's setup, dataset, and hook-function cells.
# The saved rows preserve the selected image order across all nine stages.
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
pairs, _ = observed_modules('ResNet-18', model)
feats = extract_blocks(model, loader, pairs)
del model
stage_names = np.array(list(feats))
names = np.array([raw.names[i] for i in indices])
assert len(stage_names) == 9 and len(names) == len(labels) == 200
payload = {'stages': stage_names, 'labels': labels, 'archive_paths': names,
           'selected_indices': np.array(indices)}
for j, stage in enumerate(stage_names):
    payload[f'features_{j}'] = feats[stage].astype(np.float32)
out = 'resnet18_per_image_vectors.npz'
np.savez_compressed(out, **payload)
print('Saved', out, 'stages:', list(stage_names),
      'shapes:', [feats[s].shape for s in stage_names],
      'label counts:', np.unique(labels, return_counts=True))


Saved resnet18_per_image_vectors.npz stages: [np.str_('stem'), np.str_('s1.b1'), np.str_('s1.b2'), np.str_('s2.b1'), np.str_('s2.b2'), np.str_('s3.b1'), np.str_('s3.b2'), np.str_('s4.b1'), np.str_('s4.b2')] shapes: [(200, 64), (200, 64), (200, 64), (200, 128), (200, 128), (200, 256), (200, 256), (200, 512), (200, 512)] label counts: (array([3, 5]), array([100, 100]))


In [5]:
# Run after setup, dataset, and hook-function cells in this notebook.
# The sample order and preprocessing are identical to the ResNet-18 export.
model152 = models.resnet152(weights=models.ResNet152_Weights.IMAGENET1K_V2)
pairs152, _ = observed_modules('ResNet-152', model152)
feats152 = extract_blocks(model152, loader, pairs152)
del model152
stages152 = np.array(list(feats152))
assert len(stages152) == 51 and len(labels) == 200
payload152 = {
    'stages': stages152,
    'labels': labels,
    'archive_paths': np.array([raw.names[i] for i in indices]),
    'selected_indices': np.array(indices),
}
for j, stage in enumerate(stages152):
    payload152[f'features_{j}'] = feats152[stage].astype(np.float32)
out152 = 'resnet152_per_image_vectors.npz'
np.savez_compressed(out152, **payload152)
print('Saved', out152, 'points:', len(stages152),
      'first/last:', stages152[0], stages152[-1],
      'feature widths:', sorted(set(x.shape[1] for x in feats152.values())),
      'label counts:', np.unique(labels, return_counts=True))
del feats152, payload152
gc.collect()


Saved resnet152_per_image_vectors.npz points: 51 first/last: stem s4.b3 feature widths: [64, 256, 512, 1024, 2048] label counts: (array([3, 5]), array([100, 100]))


37627

In [6]:
# Same selected 200 images and transform as the ResNet experiments.
model_tiny = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
pairs_tiny, _ = observed_modules('ConvNeXt-Tiny', model_tiny)
feats_tiny = extract_blocks(model_tiny, loader, pairs_tiny)
del model_tiny
stages_tiny = np.array(list(feats_tiny))
assert len(stages_tiny) == 22 and len(labels) == 200
payload_tiny = {
    'stages': stages_tiny,
    'labels': labels,
    'archive_paths': np.array([raw.names[i] for i in indices]),
    'selected_indices': np.array(indices),
}
for j, stage in enumerate(stages_tiny):
    payload_tiny[f'features_{j}'] = feats_tiny[stage].astype(np.float32)
out_tiny = 'convnext_tiny_per_image_vectors.npz'
np.savez_compressed(out_tiny, **payload_tiny)
print('Saved', out_tiny, 'points:', len(stages_tiny),
      'stages:', list(stages_tiny),
      'feature widths:', sorted(set(x.shape[1] for x in feats_tiny.values())),
      'label counts:', np.unique(labels, return_counts=True))
del feats_tiny, payload_tiny
gc.collect()


Saved convnext_tiny_per_image_vectors.npz points: 22 stages: [np.str_('stem'), np.str_('s1.b1'), np.str_('s1.b2'), np.str_('s1.b3'), np.str_('down1'), np.str_('s2.b1'), np.str_('s2.b2'), np.str_('s2.b3'), np.str_('down2'), np.str_('s3.b1'), np.str_('s3.b2'), np.str_('s3.b3'), np.str_('s3.b4'), np.str_('s3.b5'), np.str_('s3.b6'), np.str_('s3.b7'), np.str_('s3.b8'), np.str_('s3.b9'), np.str_('down3'), np.str_('s4.b1'), np.str_('s4.b2'), np.str_('s4.b3')] feature widths: [96, 192, 384, 768] label counts: (array([3, 5]), array([100, 100]))


10

In [7]:
# Same selected 200 images and transform as the ResNet experiments.
model_base = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
pairs_base, _ = observed_modules('ConvNeXt-Base', model_base)
feats_base = extract_blocks(model_base, loader, pairs_base)
del model_base
stages_base = np.array(list(feats_base))
assert len(stages_base) == 40 and len(labels) == 200
payload_base = {
    'stages': stages_base,
    'labels': labels,
    'archive_paths': np.array([raw.names[i] for i in indices]),
    'selected_indices': np.array(indices),
}
for j, stage in enumerate(stages_base):
    payload_base[f'features_{j}'] = feats_base[stage].astype(np.float32)
out_base = 'convnext_base_per_image_vectors.npz'
np.savez_compressed(out_base, **payload_base)
print('Saved', out_base, 'points:', len(stages_base),
      'stages:', list(stages_base),
      'feature widths:', sorted(set(x.shape[1] for x in feats_base.values())),
      'label counts:', np.unique(labels, return_counts=True))
del feats_base, payload_base
gc.collect()


Saved convnext_base_per_image_vectors.npz points: 40 stages: [np.str_('stem'), np.str_('s1.b1'), np.str_('s1.b2'), np.str_('s1.b3'), np.str_('down1'), np.str_('s2.b1'), np.str_('s2.b2'), np.str_('s2.b3'), np.str_('down2'), np.str_('s3.b1'), np.str_('s3.b2'), np.str_('s3.b3'), np.str_('s3.b4'), np.str_('s3.b5'), np.str_('s3.b6'), np.str_('s3.b7'), np.str_('s3.b8'), np.str_('s3.b9'), np.str_('s3.b10'), np.str_('s3.b11'), np.str_('s3.b12'), np.str_('s3.b13'), np.str_('s3.b14'), np.str_('s3.b15'), np.str_('s3.b16'), np.str_('s3.b17'), np.str_('s3.b18'), np.str_('s3.b19'), np.str_('s3.b20'), np.str_('s3.b21'), np.str_('s3.b22'), np.str_('s3.b23'), np.str_('s3.b24'), np.str_('s3.b25'), np.str_('s3.b26'), np.str_('s3.b27'), np.str_('down3'), np.str_('s4.b1'), np.str_('s4.b2'), np.str_('s4.b3')] feature widths: [128, 256, 512, 1024] label counts: (array([3, 5]), array([100, 100]))


10

## Metric definitions
Raw pooled features for CKA and Fisher; per-image unit features for cosine geometry and PR. Probe uses held-out splits with preprocessing fitted on training rows. Its split SD is separate from image-bootstrap uncertainty.


In [9]:
# Metrics: cosine pairwise geometry on per-image unit vectors; raw Fisher ratio.
# Squared Euclidean distance on unit vectors is 2 * cosine distance, so these are not independent measurements.
def representation_stats(x,y):
    x=np.asarray(x,dtype=np.float64)
    z=x/np.maximum(np.linalg.norm(x,axis=1,keepdims=True),1e-12)
    cat,dog=z[y==3],z[y==5]
    def within_cos(q):
        n=len(q)
        return 1-(np.sum(q.sum(axis=0)**2)-n)/(n*(n-1))
    dc=(within_cos(cat)+within_cos(dog))/2
    db=1-cat.mean(axis=0)@dog.mean(axis=0)
    # For mean *Euclidean* pairwise distances, use a 200x200 distance matrix.
    sim=np.clip(z@z.T,-1,1)
    dist=np.sqrt(np.maximum(0,2-2*sim))
    ci=np.flatnonzero(y==3); di=np.flatnonzero(y==5)
    wc=dist[np.ix_(ci,ci)][np.triu_indices(len(ci),1)].mean()
    wd=dist[np.ix_(di,di)][np.triu_indices(len(di),1)].mean()
    edw=(wc+wd)/2; edb=dist[np.ix_(ci,di)].mean()
    # Fisher on raw, globally pooled features, invariant to a global scalar only.
    a,b=x[y==3],x[y==5]; ma,mb=a.mean(0),b.mean(0)
    var=np.mean(np.sum((a-ma)**2,axis=1))+np.mean(np.sum((b-mb)**2,axis=1))
    fisher=np.sum((ma-mb)**2)/max(var,1e-20)
    centered=z-z.mean(0)
    eigen=np.maximum(np.linalg.eigvalsh(centered@centered.T),0)
    pr=eigen.sum()**2/max(np.square(eigen).sum(),1e-20)
    return {'d_within_cos':dc,'d_between_cos':db,'S':db/dc,
            'd_within_euclid_unit':edw,'d_between_euclid_unit':edb,
            'Fisher_raw':fisher,'PR':pr,'mean_raw_norm':np.linalg.norm(x,axis=1).mean()}

def centered_gram(x):
    x=np.asarray(x,dtype=np.float64)
    x=x-x.mean(0,keepdims=True)
    gram=x@x.T
    return gram/np.maximum(np.linalg.norm(gram,'fro'),1e-20)

def linear_cka(x1,x2):
    return float(np.sum(centered_gram(x1)*centered_gram(x2)))

def probe_scores(x,y,splits):
    acc=[]
    for train,test in splits:
        p=make_pipeline(Normalizer(),StandardScaler(),LogisticRegression(C=1,max_iter=2000,solver='liblinear',random_state=SEED))
        p.fit(x[train],y[train]); acc.append(accuracy_score(y[test],p.predict(x[test])))
    return float(np.mean(acc)),float(np.std(acc,ddof=1))

def relative_geometry_trajectory(feats,y):
    # Shared coordinates: each sample's similarities to the SAME ordered set of images.
    # Center and unit-normalize each sample Gram matrix before pooling all class centroids in a single PCA.
    points=[]
    for x in feats.values():
        z=x.astype(np.float64); z/=np.maximum(np.linalg.norm(z,axis=1,keepdims=True),1e-12)
        k=centered_gram(z)
        points.extend([k[y==3].mean(0),k[y==5].mean(0)])
    return PCA(n_components=2).fit_transform(np.asarray(points))


In [10]:
"""Paired, class-stratified image bootstrap for adjacent biased linear CKA.

Reuses raw pooled feature caches. Intervals are pointwise 10–90 percentiles
(central 80%), not simultaneous bands or causal tests. Probe split SD is
reported separately and is never called image-bootstrap uncertainty.
"""
from pathlib import Path
import re
import numpy as np
import pandas as pd

CACHE_FILES = {
    'ResNet-18': 'resnet18_per_image_vectors.npz',
    'ResNet-152': 'resnet152_per_image_vectors.npz',
    'ConvNeXt-Tiny': 'convnext_tiny_per_image_vectors.npz',
    'ConvNeXt-Base': 'convnext_base_per_image_vectors.npz',
}
METRICS = ['CKA_prev', 'd_within_cos', 'd_between_cos', 'S',
           'd_within_euclid_unit', 'd_between_euclid_unit', 'Fisher_raw',
           'PR', 'mean_raw_norm', 'probe_mean', 'probe_sd']


def stratified_indices(labels, repeats=1000, seed=20261006):
    labels = np.asarray(labels)
    if labels.ndim != 1 or len(np.unique(labels)) < 2 or repeats < 1:
        raise ValueError('Need a label vector, two classes, and positive repeats.')
    rng = np.random.default_rng(seed)
    groups = [np.flatnonzero(labels == c) for c in np.unique(labels)]
    return np.concatenate([rng.choice(g, (repeats, len(g)), replace=True)
                           for g in groups], axis=1)


def center_gram(gram):
    return gram - gram.mean(0, keepdims=True) - gram.mean(1, keepdims=True) + gram.mean()


def cka_curve(grams, indices):
    """Recenter AFTER row/column resampling, including duplicated observations."""
    result = np.full(len(grams), np.nan)
    previous = None
    for i, gram in enumerate(grams):
        k = center_gram(gram[np.ix_(indices, indices)])
        norm = np.sqrt(np.sum(k * k))
        if norm <= 1e-20:
            raise ValueError('Degenerate representation: CKA is undefined.')
        k = k / norm
        if previous is not None:
            result[i] = np.sum(previous * k)
        previous = k
    return result


def stage_contrasts(stages):
    """First-block adjacent CKA vs later block adjacencies in the same stage.

    ConvNeXt's downsample adjacency is separately visible in the full curve;
    sK.b1 measures downsample output -> first block, unlike ResNet sK.b1.
    This contrast reduces but does not eliminate depth/architecture confounds.
    """
    groups = {}
    for i, name in enumerate(stages):
        match = re.fullmatch(r's(\d+)\.b(\d+)', str(name))
        if match:
            stage, block = map(int, match.groups())
            groups.setdefault(stage, {})[block] = i
    return [(s, blocks[1], [blocks[b] for b in sorted(blocks) if b > 1])
            for s, blocks in sorted(groups.items()) if 1 in blocks and len(blocks) > 1]


def bootstrap_caches(cache_dir='.', repeats=1000, seed=20261006, progress=print):
    reference = None
    summaries, samples, dips, dip_samples = [], [], [], []
    draw_indices = None
    for model, filename in CACHE_FILES.items():
        path = Path(cache_dir) / filename
        with np.load(path, allow_pickle=False) as data:
            labels = data['labels'].copy()
            paths = data['archive_paths'].copy()
            stages = data['stages'].copy()
            if reference is None:
                reference = (labels, paths)
                if len(paths) != len(set(paths.tolist())):
                    raise ValueError('Sample cache contains duplicate source images.')
                draw_indices = stratified_indices(labels, repeats, seed)
            elif not (np.array_equal(labels, reference[0]) and np.array_equal(paths, reference[1])):
                raise ValueError(f'Mismatched image identities/order: {filename}')
            grams = []
            for i in range(len(stages)):
                x = np.asarray(data[f'features_{i}'], dtype=np.float64)
                if x.ndim != 2 or len(x) != len(labels) or not np.isfinite(x).all():
                    raise ValueError(f'Invalid features: {filename}, layer {i}')
                # Translation invariance allows this precentering for precision.
                x -= x.mean(0, keepdims=True)
                grams.append(x @ x.T)
        original = cka_curve(grams, np.arange(len(labels)))
        boot = np.empty((repeats, len(stages)))
        for b, idx in enumerate(draw_indices):
            boot[b] = cka_curve(grams, idx)
            if progress and ((b + 1) % 100 == 0 or b + 1 == repeats):
                progress(f'{model}: bootstrap {b + 1}/{repeats}', flush=True)
        for depth, stage in enumerate(stages):
            values = boot[:, depth]
            q = [np.nan] * 3 if depth == 0 else np.percentile(values, [10, 50, 90])
            summaries.append(dict(model=model, depth=depth, stage=stage,
                                  estimate=original[depth], p10=q[0], median=q[1], p90=q[2],
                                  n_images=len(labels), n_bootstrap=repeats, seed=seed))
            samples.extend(dict(model=model, replicate=b, depth=depth, stage=stage,
                                CKA_prev=value) for b, value in enumerate(values))
        for stage, boundary, later in stage_contrasts(stages):
            delta = np.median(boot[:, later], axis=1) - boot[:, boundary]
            q = np.percentile(delta, [10, 50, 90])
            dips.append(dict(model=model, stage=f's{stage}', boundary_block=stages[boundary],
                             comparator_blocks=';'.join(stages[later]), n_comparators=len(later),
                             estimate=np.median(original[later]) - original[boundary],
                             p10=q[0], median=q[1], p90=q[2],
                             fraction_positive=float(np.mean(delta > 0)),
                             n_bootstrap=repeats, seed=seed))
            dip_samples.extend(dict(model=model, stage=f's{stage}', replicate=b, delta=value)
                               for b, value in enumerate(delta))
    return { 'CKA_prev_bootstrap': pd.DataFrame(summaries),
             'CKA_prev_bootstrap_replicates': pd.DataFrame(samples),
             'boundary_dip': pd.DataFrame(dips),
             'boundary_dip_replicates': pd.DataFrame(dip_samples),
             'bootstrap_indices': pd.DataFrame(draw_indices),
             'sample_manifest': pd.DataFrame({'row': np.arange(len(reference[0])),
                                             'label': reference[0], 'archive_path': reference[1]}) }


def export_metric_csvs(results, output_dir):
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    keys = ['model', 'depth', 'stage', 'boundary']
    for metric in METRICS:
        results[keys + [metric]].to_csv(output / f'{metric}.csv', index=False)


In [11]:
splits=list(StratifiedShuffleSplit(n_splits=PROBE_SPLITS,test_size=.30,random_state=SEED).split(np.zeros(len(labels)),labels))
all_rows=[]; trajectories={}; model_boundaries={}; model_stages={}
for name,constructor,weights in MODEL_SPECS:
    print('Starting',name,flush=True)
    with np.load(CACHE_FILES[name], allow_pickle=False) as data:
        assert np.array_equal(data['labels'], labels)
        assert np.array_equal(data['archive_paths'], np.array([raw.names[i] for i in indices]))
        feats = {str(stage): data[f'features_{j}'].copy() for j, stage in enumerate(data['stages'])}
    boundaries = [stage for stage in feats if stage == 'stem' or stage.startswith('down') or stage.endswith('.b1')]
    pairs = list(feats.items())
    model_boundaries[name]=boundaries
    model_stages[name]=list(feats)
    trajectories[name]=relative_geometry_trajectory(feats,labels)
    previous=None
    for depth,(stage,x) in enumerate(feats.items()):
        stats=representation_stats(x,labels)
        mean,sd=probe_scores(x,labels,splits)
        all_rows.append({'model':name,'depth':depth,'stage':stage,'boundary':stage in boundaries,
                         'CKA_prev':np.nan if previous is None else linear_cka(previous,x),
                         'probe_mean':mean,'probe_sd':sd,**stats})
        previous=x
    del feats,previous
    print('Finished',name,'measurement points',len(pairs),flush=True)
results=pd.DataFrame(all_rows)
display(results.groupby('model',sort=False).agg(points=('stage','size'),
    min_adjacent_CKA=('CKA_prev','min'),S_first=('S','first'),S_last=('S','last'),
    probe_first=('probe_mean','first'),probe_last=('probe_mean','last')).round(3))


Starting ResNet-18
Finished ResNet-18 measurement points 9
Starting ResNet-152
Finished ResNet-152 measurement points 51
Starting ConvNeXt-Tiny
Finished ConvNeXt-Tiny measurement points 22
Starting ConvNeXt-Base
Finished ConvNeXt-Base measurement points 40


,points,min_adjacent_CKA,S_first,S_last,probe_first,probe_last
model,,,,,,
ResNet-18,9,0.761,1.016,1.342,0.577,0.997
ResNet-152,51,0.396,1.004,1.305,0.613,1.000
ConvNeXt-Tiny,22,0.118,1.033,1.341,0.543,1.000
ConvNeXt-Base,40,0.088,1.030,1.347,0.530,1.000


## Paired stratified bootstrap of adjacent CKA

Sample each observed class with replacement, preserving its original count. All models/layers reuse the same draw. Gram matrices are resampled on BOTH axes and recentered for each draw, preserving paired image identities. No re-training or new forward passes.


In [13]:
bootstrap = bootstrap_caches('.', BOOTSTRAP_REPEATS, BOOTSTRAP_SEED)
for filename, frame in bootstrap.items():
    frame.to_csv(RESULT_DIR / f'{filename}.csv', index=False)
display(bootstrap['CKA_prev_bootstrap'].head())


ResNet-18: bootstrap 100/1000
ResNet-18: bootstrap 200/1000
ResNet-18: bootstrap 300/1000
ResNet-18: bootstrap 400/1000
ResNet-18: bootstrap 500/1000
ResNet-18: bootstrap 600/1000
ResNet-18: bootstrap 700/1000
ResNet-18: bootstrap 800/1000
ResNet-18: bootstrap 900/1000
ResNet-18: bootstrap 1000/1000
ResNet-152: bootstrap 100/1000
ResNet-152: bootstrap 200/1000
ResNet-152: bootstrap 300/1000
ResNet-152: bootstrap 400/1000
ResNet-152: bootstrap 500/1000
ResNet-152: bootstrap 600/1000
ResNet-152: bootstrap 700/1000
ResNet-152: bootstrap 800/1000
ResNet-152: bootstrap 900/1000
ResNet-152: bootstrap 1000/1000
ConvNeXt-Tiny: bootstrap 100/1000
ConvNeXt-Tiny: bootstrap 200/1000
ConvNeXt-Tiny: bootstrap 300/1000
ConvNeXt-Tiny: bootstrap 400/1000
ConvNeXt-Tiny: bootstrap 500/1000
ConvNeXt-Tiny: bootstrap 600/1000
ConvNeXt-Tiny: bootstrap 700/1000
ConvNeXt-Tiny: bootstrap 800/1000
ConvNeXt-Tiny: bootstrap 900/1000
ConvNeXt-Tiny: bootstrap 1000/1000
ConvNeXt-Base: bootstrap 100/1000
ConvNeXt-Base

,model,depth,stage,estimate,p10,median,p90,n_images,n_bootstrap,seed
0,ResNet-18,0,stem,NaN,NaN,NaN,NaN,200,1000,20261006
1,ResNet-18,1,s1.b1,0.887812,0.877321,0.889586,0.902827,200,1000,20261006
2,ResNet-18,2,s1.b2,0.839259,0.827731,0.845325,0.861956,200,1000,20261006
3,ResNet-18,3,s2.b1,0.829638,0.805239,0.838389,0.861133,200,1000,20261006
4,ResNet-18,4,s2.b2,0.913877,0.914285,0.921322,0.928837,200,1000,20261006


## CKA prev



In [15]:
# CKA prev: one independent figure per architecture.
results[['model','depth','stage','boundary','CKA_prev']].to_csv(RESULT_DIR / 'CKA_prev.csv', index=False)
for name, g in results.groupby('model', sort=False):
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(g.depth, g['CKA_prev'], '.-', label='Original sample')
    band = bootstrap['CKA_prev_bootstrap'].query('model == @name').sort_values('depth')
    assert np.array_equal(g.depth.to_numpy(), band.depth.to_numpy())
    ax.fill_between(band.depth.to_numpy(), band.p10.to_numpy(), band.p90.to_numpy(), alpha=.2,
                    label='Pointwise bootstrap p10–p90 (80%)')
    for depth in g.loc[g.boundary & (g.depth > 0), 'depth']:
        ax.axvline(depth, color='gray', linestyle=':', alpha=.35)
    ax.set(title=name + ' — CKA prev', xlabel='Observed layer / transition index', ylabel='CKA_prev')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_CKA_prev.png'), dpi=160)
    plt.show()


## d within cos

[resnet18](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet18/d_within_cos.csv) | [resnet152](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet152/d_within_cos.csv) | [convnext_tiny](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_tiny/d_within_cos.csv) | [convnext_base](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_base/d_within_cos.csv)


In [17]:
# d within cos: one independent figure per architecture.
results[['model','depth','stage','boundary','d_within_cos']].to_csv(RESULT_DIR / 'd_within_cos.csv', index=False)
for name, g in results.groupby('model', sort=False):
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(g.depth, g['d_within_cos'], '.-', label='Original sample')
    for depth in g.loc[g.boundary & (g.depth > 0), 'depth']:
        ax.axvline(depth, color='gray', linestyle=':', alpha=.35)
    ax.set(title=name + ' — d within cos', xlabel='Observed layer / transition index', ylabel='d_within_cos')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_d_within_cos.png'), dpi=160)
    plt.show()


## d between cos

[resnet18](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet18/d_between_cos.csv) | [resnet152](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet152/d_between_cos.csv) | [convnext_tiny](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_tiny/d_between_cos.csv) | [convnext_base](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_base/d_between_cos.csv)


In [19]:
# d between cos: one independent figure per architecture.
results[['model','depth','stage','boundary','d_between_cos']].to_csv(RESULT_DIR / 'd_between_cos.csv', index=False)
for name, g in results.groupby('model', sort=False):
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(g.depth, g['d_between_cos'], '.-', label='Original sample')
    for depth in g.loc[g.boundary & (g.depth > 0), 'depth']:
        ax.axvline(depth, color='gray', linestyle=':', alpha=.35)
    ax.set(title=name + ' — d between cos', xlabel='Observed layer / transition index', ylabel='d_between_cos')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_d_between_cos.png'), dpi=160)
    plt.show()


## S

[resnet18](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet18/S.csv) | [resnet152](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet152/S.csv) | [convnext_tiny](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_tiny/S.csv) | [convnext_base](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_base/S.csv)


In [21]:
# S: one independent figure per architecture.
results[['model','depth','stage','boundary','S']].to_csv(RESULT_DIR / 'S.csv', index=False)
for name, g in results.groupby('model', sort=False):
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(g.depth, g['S'], '.-', label='Original sample')
    for depth in g.loc[g.boundary & (g.depth > 0), 'depth']:
        ax.axvline(depth, color='gray', linestyle=':', alpha=.35)
    ax.set(title=name + ' — S', xlabel='Observed layer / transition index', ylabel='S')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_S.png'), dpi=160)
    plt.show()


## d within euclid unit

[resnet18](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet18/d_within_euclid_unit.csv) | [resnet152](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet152/d_within_euclid_unit.csv) | [convnext_tiny](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_tiny/d_within_euclid_unit.csv) | [convnext_base](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_base/d_within_euclid_unit.csv)


In [23]:
# d within euclid unit: one independent figure per architecture.
results[['model','depth','stage','boundary','d_within_euclid_unit']].to_csv(RESULT_DIR / 'd_within_euclid_unit.csv', index=False)
for name, g in results.groupby('model', sort=False):
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(g.depth, g['d_within_euclid_unit'], '.-', label='Original sample')
    for depth in g.loc[g.boundary & (g.depth > 0), 'depth']:
        ax.axvline(depth, color='gray', linestyle=':', alpha=.35)
    ax.set(title=name + ' — d within euclid unit', xlabel='Observed layer / transition index', ylabel='d_within_euclid_unit')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_d_within_euclid_unit.png'), dpi=160)
    plt.show()


## d between euclid unit

[resnet18](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet18/d_between_euclid_unit.csv) | [resnet152](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet152/d_between_euclid_unit.csv) | [convnext_tiny](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_tiny/d_between_euclid_unit.csv) | [convnext_base](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_base/d_between_euclid_unit.csv)


In [25]:
# d between euclid unit: one independent figure per architecture.
results[['model','depth','stage','boundary','d_between_euclid_unit']].to_csv(RESULT_DIR / 'd_between_euclid_unit.csv', index=False)
for name, g in results.groupby('model', sort=False):
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(g.depth, g['d_between_euclid_unit'], '.-', label='Original sample')
    for depth in g.loc[g.boundary & (g.depth > 0), 'depth']:
        ax.axvline(depth, color='gray', linestyle=':', alpha=.35)
    ax.set(title=name + ' — d between euclid unit', xlabel='Observed layer / transition index', ylabel='d_between_euclid_unit')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_d_between_euclid_unit.png'), dpi=160)
    plt.show()


## Fisher raw

[resnet18](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet18/Fisher_raw.csv) | [resnet152](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet152/Fisher_raw.csv) | [convnext_tiny](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_tiny/Fisher_raw.csv) | [convnext_base](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_base/Fisher_raw.csv)


In [27]:
# Fisher raw: one independent figure per architecture.
results[['model','depth','stage','boundary','Fisher_raw']].to_csv(RESULT_DIR / 'Fisher_raw.csv', index=False)
for name, g in results.groupby('model', sort=False):
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(g.depth, g['Fisher_raw'], '.-', label='Original sample')
    for depth in g.loc[g.boundary & (g.depth > 0), 'depth']:
        ax.axvline(depth, color='gray', linestyle=':', alpha=.35)
    ax.set(title=name + ' — Fisher raw', xlabel='Observed layer / transition index', ylabel='Fisher_raw')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_Fisher_raw.png'), dpi=160)
    plt.show()


## PR

[resnet18](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet18/PR.csv) | [resnet152](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet152/PR.csv) | [convnext_tiny](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_tiny/PR.csv) | [convnext_base](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_base/PR.csv)


In [29]:
# PR: one independent figure per architecture.
results[['model','depth','stage','boundary','PR']].to_csv(RESULT_DIR / 'PR.csv', index=False)
for name, g in results.groupby('model', sort=False):
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(g.depth, g['PR'], '.-', label='Original sample')
    for depth in g.loc[g.boundary & (g.depth > 0), 'depth']:
        ax.axvline(depth, color='gray', linestyle=':', alpha=.35)
    ax.set(title=name + ' — PR', xlabel='Observed layer / transition index', ylabel='PR')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_PR.png'), dpi=160)
    plt.show()


## mean raw norm

[resnet18](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet18/mean_raw_norm.csv) | [resnet152](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet152/mean_raw_norm.csv) | [convnext_tiny](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_tiny/mean_raw_norm.csv) | [convnext_base](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_base/mean_raw_norm.csv)


In [31]:
# mean raw norm: one independent figure per architecture.
results[['model','depth','stage','boundary','mean_raw_norm']].to_csv(RESULT_DIR / 'mean_raw_norm.csv', index=False)
for name, g in results.groupby('model', sort=False):
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(g.depth, g['mean_raw_norm'], '.-', label='Original sample')
    for depth in g.loc[g.boundary & (g.depth > 0), 'depth']:
        ax.axvline(depth, color='gray', linestyle=':', alpha=.35)
    ax.set(title=name + ' — mean raw norm', xlabel='Observed layer / transition index', ylabel='mean_raw_norm')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_mean_raw_norm.png'), dpi=160)
    plt.show()


## probe mean

[resnet18](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet18/probe_mean.csv) | [resnet152](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet152/probe_mean.csv) | [convnext_tiny](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_tiny/probe_mean.csv) | [convnext_base](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_base/probe_mean.csv)


In [33]:
# probe mean: one independent figure per architecture.
results[['model','depth','stage','boundary','probe_mean']].to_csv(RESULT_DIR / 'probe_mean.csv', index=False)
for name, g in results.groupby('model', sort=False):
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(g.depth, g['probe_mean'], '.-', label='Original sample')
    for depth in g.loc[g.boundary & (g.depth > 0), 'depth']:
        ax.axvline(depth, color='gray', linestyle=':', alpha=.35)
    ax.set(title=name + ' — probe mean', xlabel='Observed layer / transition index', ylabel='probe_mean')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_probe_mean.png'), dpi=160)
    plt.show()


## probe sd

[resnet18](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet18/probe_sd.csv) | [resnet152](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet152/probe_sd.csv) | [convnext_tiny](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_tiny/probe_sd.csv) | [convnext_base](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_base/probe_sd.csv)


In [35]:
# probe sd: one independent figure per architecture.
results[['model','depth','stage','boundary','probe_sd']].to_csv(RESULT_DIR / 'probe_sd.csv', index=False)
for name, g in results.groupby('model', sort=False):
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(g.depth, g['probe_sd'], '.-', label='Original sample')
    for depth in g.loc[g.boundary & (g.depth > 0), 'depth']:
        ax.axvline(depth, color='gray', linestyle=':', alpha=.35)
    ax.set(title=name + ' — probe sd', xlabel='Observed layer / transition index', ylabel='probe_sd')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_probe_sd.png'), dpi=160)
    plt.show()


## Similarity-fingerprint centroid trajectories

Coordinates use a shared PCA within each model and are not comparable between models. [resnet18](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet18/similarity_fingerprint_trajectory.csv) | [resnet152](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/resnet152/similarity_fingerprint_trajectory.csv) | [convnext_tiny](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_tiny/similarity_fingerprint_trajectory.csv) | [convnext_base](https://github.com/Lawson-Dong/representation-alignment-/blob/representation-vector-geometric-dynamics/experiments/geometric_dynamics/results/geometry/convnext_base/similarity_fingerprint_trajectory.csv)


In [44]:
trajectory_rows = []
for name, coords in trajectories.items():
    g = results.query('model == @name').sort_values('depth')
    fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
    for offset, category in [(0, 'cat'), (1, 'dog')]:
        xy = coords[offset::2]
        ax.plot(xy[:, 0], xy[:, 1], '-o', markersize=3, label=category + ' centroid')
        for j, row in enumerate(g.itertuples()):
            trajectory_rows.append(dict(model=name, depth=row.depth, stage=row.stage, category=category,
                                        pca1=xy[j, 0], pca2=xy[j, 1]))
    ax.set(title=name + ' — similarity-fingerprint trajectory', xlabel='PCA 1', ylabel='PCA 2')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_similarity_fingerprint_trajectory.png'), dpi=160)
    plt.show()
pd.DataFrame(trajectory_rows).to_csv(RESULT_DIR / 'similarity_fingerprint_trajectory.csv', index=False)


## Within-stage boundary dip

Δ = median(adjacent CKA of later blocks in this stage) − adjacent CKA at its first block. Exclude stem and standalone downsample points from this contrast. Compare separately for each stage and model.

**ConvNeXt caveat:** its first-block CKA compares downsample output with the first block; ResNet's compares the previous stage output with the first block. The full CKA plots show standalone downsampling changes separately. Within-stage comparisons reduce but do not eliminate depth confounding and do not isolate a causal downsampling effect.



In [37]:
dip = bootstrap['boundary_dip']
display(dip)
for name, g in dip.groupby('model', sort=False):
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    y = np.arange(len(g))
    ax.hlines(y, g.p10, g.p90, linewidth=3, label='Bootstrap p10–p90 (80%)')
    ax.scatter(g.estimate, y, label='Original Δ', zorder=3)
    ax.axvline(0, color='gray', linestyle=':')
    ax.set_yticks(y, g.stage)
    ax.set(title=name + ' — within-stage dip', xlabel='Median later-block CKA − first-block CKA')
    ax.legend()
    fig.savefig(RESULT_DIR / (name + '_boundary_dip.png'), dpi=160)
    plt.show()


,model,stage,boundary_block,comparator_blocks,n_comparators,estimate,p10,median,p90,fraction_positive,n_bootstrap,seed
0,ResNet-18,s1,s1.b1,s1.b2,1,-0.048554,-0.061582,-0.044309,-0.028874,0.001,1000,20261006
1,ResNet-18,s2,s2.b1,s2.b2,1,0.084240,0.058566,0.081359,0.122142,1.000,1000,20261006
2,ResNet-18,s3,s3.b1,s3.b2,1,0.046443,0.027548,0.042801,0.058094,1.000,1000,20261006
3,ResNet-18,s4,s4.b1,s4.b2,1,0.033019,-0.000624,0.023556,0.050308,0.895,1000,20261006
4,ResNet-152,s1,s1.b1,s1.b2;s1.b3,2,0.597970,0.550670,0.587580,0.623926,1.000,1000,20261006
5,ResNet-152,s2,s2.b1,s2.b2;s2.b3;s2.b4;s2.b5;s2.b6;s2.b7;s2.b8,7,0.026993,0.021428,0.027394,0.033781,1.000,1000,20261006
6,ResNet-152,s3,s3.b1,s3.b2;s3.b3;s3.b4;s3.b5;s3.b6;s3.b7;s3.b8;s3.b...,35,0.064389,0.049533,0.057969,0.066488,1.000,1000,20261006
7,ResNet-152,s4,s4.b1,s4.b2;s4.b3,2,0.024406,0.012376,0.019344,0.027776,1.000,1000,20261006
8,ConvNeXt-Tiny,s1,s1.b1,s1.b2;s1.b3,2,-0.195006,-0.205581,-0.189118,-0.174652,0.000,1000,20261006
9,ConvNeXt-Tiny,s2,s2.b1,s2.b2;s2.b3,2,-0.077219,-0.105376,-0.076492,-0.043510,0.000,1000,20261006


## Interpretation and caveats

- **Linear CKA** compares centered sample Gram matrices and handles unequal feature widths. Large adjacent CKA means similar relational geometry on these 200 examples, not identical activations. A dip can reflect a downsampling or pooling change; it is not by itself a phase transition. Compare stage boundaries with within-stage blocks. See [Kornblith et al., ICML 2019](https://proceedings.mlr.press/v97/kornblith19a.html).
- **S** uses cosine distance after per-image L2 normalization. A rising S may accompany rising within distance; inspect both. Unit-vector Euclidean distance is related by $\|u-v\|=\sqrt{2(1-\cos	heta)}$; it is another scale of the same pairwise geometry, not an independent signal.
- **Fisher** here uses squared class-centroid separation divided by the sum of raw within-class mean squared spreads. Its raw feature norm makes it a different lens from cosine S, but class separation metrics remain correlated.
- **PR** is the participation ratio of the centered, unit-normalized sample cloud. It is an effective dimension under this estimator, not a unique intrinsic manifold dimension.
- **Trajectory** uses each image's similarity to the same ordered 200 images, then one shared PCA over all class-centroid similarity fingerprints *within each model*. This permits plotted paths when channel counts change, but coordinates describe changing relational similarity, not raw activation trajectories. 2D projection can hide changes.
- **Probe** freezes every network and fits the same regularized linear classifier on five paired 70/30 splits. Test identities match across all layers and architectures. The reported SD is descriptive; splits overlap.
- Global average pooling at every hook discards spatial information. These are block outputs, with architectural transitions marked; the horizontal coordinate counts **observations**, not equal physical depth. Cross-architecture curves cannot be treated as a controlled depth-only comparison, since weights and network designs also change.


## Export and verify the completed run

The folder contains one CSV per metric, bootstrap intervals/replicates, matched-image manifest, draw indices, figures and a run record. Probe is NOT recomputed on duplicated bootstrap rows, which would risk train/test image leakage.


In [40]:
import json, platform, shutil
export_metric_csvs(results, RESULT_DIR)
assert len(results) == 122
assert all(np.isfinite(results.loc[results.depth > 0, 'CKA_prev']))
for name, g in results.groupby('model', sort=False):
    check = bootstrap['CKA_prev_bootstrap'].query('model == @name')
    assert np.allclose(g.CKA_prev, check.estimate, equal_nan=True, atol=1e-10)
record = dict(status='completed', bootstrap_repeats=BOOTSTRAP_REPEATS,
              bootstrap_seed=BOOTSTRAP_SEED, sample_seed=SEED,
              interval='pointwise p10-p90 (central 80%)',
              classes={str(c): int((labels == c).sum()) for c in np.unique(labels)},
              models={name: str(weights) for name, _, weights in MODEL_SPECS},
              preprocessing=str(transform), python=platform.python_version(),
              torch=torch.__version__, torchvision=torchvision.__version__,
              numpy=np.__version__, pandas=pd.__version__)
record['csv_sha256'] = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                        for p in sorted(RESULT_DIR.glob('*.csv'))}
(RESULT_DIR / 'run_record.json').write_text(json.dumps(record, indent=2))
shutil.make_archive(str(RESULT_DIR), 'zip', root_dir=RESULT_DIR.parent, base_dir=RESULT_DIR.name)
print('Verified and packed', RESULT_DIR.with_suffix('.zip'))


Verified and packed block_geometry_metrics.zip


In [45]:
from datetime import datetime, timezone
record = json.loads((RESULT_DIR / 'run_record.json').read_text())
record.update(executed_at_utc=datetime.now(timezone.utc).isoformat(),
              execution_backend='Google Colab T4',
              colab_notebook='https://colab.research.google.com/drive/11AP74Wdk1MPNdzclnGyU-d0gEBScc7NK',
              reviewed_source_commit='d111554ca72c4754f90969f54fae7f46d404db94',
              gpu=torch.cuda.get_device_name(0), dataset_md5=actual_md5,
              feature_cache_sha256={filename: hashlib.sha256(Path(filename).read_bytes()).hexdigest()
                                    for filename in CACHE_FILES.values()})
record['csv_sha256'] = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                        for p in sorted(RESULT_DIR.glob('*.csv'))}
(RESULT_DIR / 'run_record.json').write_text(json.dumps(record, indent=2))
shutil.make_archive(str(RESULT_DIR), 'zip', root_dir=RESULT_DIR.parent, base_dir=RESULT_DIR.name)
print('Completed run:', len(results), 'measurement points;', len(list(RESULT_DIR.glob('*.csv'))), 'CSV files')


Completed run: 122 measurement points; 18 CSV files


In [ ]:
# Optional download after all computations finish.
from google.colab import files
files.download(str(RESULT_DIR.with_suffix('.zip')))
